In [1]:
import glob
import os
import re
import time
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import pandas as pd
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

output_folder = "gutenberg_harvest"
os.makedirs(output_folder, exist_ok=True)

HARVEST_URL = "https://www.gutenberg.org/robot/harvest?filetypes[]=html"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AcademicResearchBot/1.0"
}

MAX_BOOKS = 20
downloaded_count = 0

print(f"Iniciando raspagem oficial via Gutenberg Harvest: {HARVEST_URL}")
print("Respeitando intervalo de 2 segundos (-w 2) conforme robot_access.html...\n")

current_harvest_url = HARVEST_URL

while current_harvest_url and downloaded_count < MAX_BOOKS:
    try:
        response = requests.get(
            current_harvest_url, headers=headers, timeout=15, verify=False
        )
        if response.status_code != 200:
            print(
                f"Erro ao acessar harvest ({response.status_code}). Interrompendo..."
            )
            break

        soup = BeautifulSoup(response.text, "html.parser")
        links = soup.find_all("a", href=True)

        file_links = []
        for l in links:
            href = l["href"]
            if (
                "aleph.gutenberg.org" in href
                or href.endswith(".zip")
                or "-h.zip" in href
            ):
                file_links.append(urljoin(current_harvest_url, href))

        for zip_url in file_links:
            if downloaded_count >= MAX_BOOKS:
                break

            # Extrai o ID apenas a partir do nome do arquivo final (ex: 1342-h.zip -> 1342)
            filename = os.path.basename(zip_url)
            match = re.search(r"(\d+)", filename)
            book_id = match.group(1) if match else f"book_{downloaded_count}"
            save_path = os.path.join(output_folder, f"{book_id}-h.zip")

            if os.path.exists(save_path):
                print(f"Livro ID {book_id} já existe localmente. Pulando...")
                continue

            time.sleep(2)
            try:
                r_file = requests.get(
                    zip_url, headers=headers, timeout=20, verify=False
                )
                if r_file.status_code == 200 and len(r_file.content) > 1000:
                    with open(save_path, "wb") as f:
                        f.write(r_file.content)
                    downloaded_count += 1
                    print(
                        f"[{downloaded_count}/{MAX_BOOKS}] Baixado com sucesso: {book_id}"
                    )
            except Exception as e_file:
                print(f"Erro no download de {zip_url}: {e_file}")

        next_page = soup.find("a", string=re.compile(r"Next\s*Page", re.I))
        if next_page and "href" in next_page.attrs:
            current_harvest_url = urljoin(HARVEST_URL, next_page["href"])
            time.sleep(2)
        else:
            current_harvest_url = None

    except Exception as e:
        print(f"Falha na execução do harvest: {e}")
        break

print(
    f"\nColeta concluída! {downloaded_count} arquivos baixados em '{output_folder}'."
)

Iniciando raspagem oficial via Gutenberg Harvest: https://www.gutenberg.org/robot/harvest?filetypes[]=html
Respeitando intervalo de 2 segundos (-w 2) conforme robot_access.html...

Livro ID 10084 já existe localmente. Pulando...
Livro ID 1554 já existe localmente. Pulando...
Livro ID 1680 já existe localmente. Pulando...
Livro ID 71 já existe localmente. Pulando...
Livro ID 1957 já existe localmente. Pulando...
Livro ID 1940 já existe localmente. Pulando...
Livro ID 1837 já existe localmente. Pulando...
Livro ID 1749 já existe localmente. Pulando...
Livro ID 245 já existe localmente. Pulando...
Livro ID 1729 já existe localmente. Pulando...
Livro ID 1925 já existe localmente. Pulando...
Livro ID 2452 já existe localmente. Pulando...
Livro ID 1715 já existe localmente. Pulando...
Livro ID 1649 já existe localmente. Pulando...
Livro ID 535 já existe localmente. Pulando...
Livro ID 2303 já existe localmente. Pulando...
Livro ID 2304 já existe localmente. Pulando...
Livro ID 430 já existe 

In [2]:
import glob
import os
import re
import string
import zipfile
from bs4 import BeautifulSoup
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

STOPWORDS_EN = set(stopwords.words("english"))
PUNCT = set(string.punctuation)


def extrair_trechos(texto, qtd_trechos=3, min_palavras=200):
  palavras = texto.split()
  total = len(palavras)
  if total < (qtd_trechos * min_palavras):
    return [" ".join(palavras)] * qtd_trechos
  marcadores = [int(total * 0.2), int(total * 0.5), int(total * 0.8)]
  return [" ".join(palavras[idx : idx + min_palavras]) for idx in marcadores]


def parse_gutenberg_html(file_path):
  with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    html_content = f.read()

  soup = BeautifulSoup(html_content, "html.parser")

  title_meta = soup.find("meta", attrs={"name": "dc.title"})
  creator_meta = soup.find("meta", attrs={"name": "dc.creator"})

  title = (
      title_meta["content"]
      if title_meta and "content" in title_meta.attrs
      else "Desconhecido"
  )
  author = (
      creator_meta["content"]
      if creator_meta and "content" in creator_meta.attrs
      else "Desconhecido"
  )

  for element in soup(["script", "style", "header", "footer"]):
    element.extract()

  text = soup.get_text(separator=" ")

  start_match = re.search(
      r"\*\*\*\s*START OF TH(IS|E) PROJECT GUTENBERG EBOOK.*?\*\*\*",
      text,
      re.IGNORECASE,
  )
  end_match = re.search(
      r"\*\*\*\s*END OF TH(IS|E) PROJECT GUTENBERG EBOOK.*?\*\*",
      text,
      re.IGNORECASE,
  )

  if start_match and end_match:
    text = text[start_match.end() : end_match.start()]
  elif start_match:
    text = text[start_match.end() :]

  clean_text = re.sub(r"\s+", " ", text).strip()
  return title, author, clean_text

In [ ]:
import glob
import os
import re
import string
import zipfile
from bs4 import BeautifulSoup
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Garante o download dos recursos do NLTK
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

STOPWORDS_EN = set(stopwords.words("english"))
PUNCT = set(string.punctuation)


def extrair_trechos(texto, qtd_trechos=3, min_palavras=200):
  palavras = texto.split()
  total = len(palavras)
  if total < (qtd_trechos * min_palavras):
    return [" ".join(palavras)] * qtd_trechos
  marcadores = [int(total * 0.2), int(total * 0.5), int(total * 0.8)]
  return [" ".join(palavras[idx : idx + min_palavras]) for idx in marcadores]


def parse_gutenberg_html_to_dataset(file_path):
  """Extrai metadados e o texto limpo diretamente do arquivo HTML do Gutenberg."""
  with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    html_content = f.read()

  soup = BeautifulSoup(html_content, "html.parser")

  # Extração de Metadados via tags <meta> do HTML
  title_meta = soup.find("meta", attrs={"name": "dc.title"})
  creator_meta = soup.find("meta", attrs={"name": "dc.creator"})
  title = (
      title_meta["content"]
      if title_meta and "content" in title_meta.attrs
      else "Desconhecido"
  )
  author = (
      creator_meta["content"]
      if creator_meta and "content" in creator_meta.attrs
      else "Desconhecido"
  )

  # Remove elementos estruturais indesejados
  for element in soup(["script", "style", "header", "footer"]):
    element.extract()

  text = soup.get_text(separator=" ")

  # Delimitadores oficiais do Project Gutenberg presentes no HTML
  start_match = re.search(
      r"\*\*\*\s*START OF TH(IS|E) PROJECT GUTENBERG EBOOK.*?\*\*\*",
      text,
      re.IGNORECASE,
  )
  end_match = re.search(
      r"\*\*\*\s*END OF TH(IS|E) PROJECT GUTENBERG EBOOK.*?\*\*",
      text,
      re.IGNORECASE,
  )

  if start_match and end_match:
    text = text[start_match.end() : end_match.start()]
  elif start_match:
    text = text[start_match.end() :]

  clean_text = re.sub(r"\s+", " ", text).strip()
  return title, author, clean_text


def build_dataset_from_harvest(base_folder):
  zip_files = glob.glob(os.path.join(base_folder, "*.zip"))
  print(f"Descompactando {len(zip_files)} arquivos .zip...")
  for zip_path in zip_files:
    try:
      with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(base_folder)
    except Exception as e:
      print(f"Erro ao descompactar {zip_path}: {e}")

  html_files = glob.glob(os.path.join(base_folder, "**/*.htm*"), recursive=True)
  print(f"Total de arquivos HTML encontrados: {len(html_files)}")

  data = []
  for file_path in html_files:
    filename = os.path.basename(file_path)
    match = re.search(r"(\d+)", filename)
    if not match:
      continue

    book_id = match.group(1)
    title, author, cleaned_text = parse_gutenberg_html_to_dataset(file_path)

    if len(cleaned_text) < 1000:
      continue

    trechos = extrair_trechos(cleaned_text, qtd_trechos=3, min_palavras=200)

    data.append({
        "book_id": book_id,
        "file_path": file_path,
        "title": title,
        "author": author,
        "text": cleaned_text,
        "trecho_1": trechos[0],
        "trecho_2": trechos[1],
        "trecho_3": trechos[2],
    })

  return pd.DataFrame(data)


# 1. Carrega e processa o dataset via HTML
df_books = build_dataset_from_harvest("gutenberg_harvest")
print(f"Livros válidos processados: {len(df_books)}")

print("Extraindo e limpando os trechos...")


def preprocess_text(text):
  if not isinstance(text, str) or not text.strip():
    return ""
  tokens = word_tokenize(text.lower(), language="english")
  tokens_filtered = [
      tok
      for tok in tokens
      if tok not in STOPWORDS_EN and not all(ch in PUNCT for ch in tok)
  ]
  return " ".join(tokens_filtered)


# 2. Processamento PLN dos trechos
for col in ["trecho_1", "trecho_2", "trecho_3"]:
  df_books[f"{col}_clean"] = df_books[col].apply(preprocess_text)

# 3. Tokenização global
df_books["tokens"] = df_books["text"].apply(
    lambda t: word_tokenize(t.lower(), language="english")
)
df_books["tokens_clean"] = df_books["tokens"].apply(
    lambda toks: [
        t for t in toks if t not in STOPWORDS_EN and not all(c in PUNCT for c in t)
    ]
)

# 4. Limpeza de colunas intermediárias do lote atual
df_novos_dados = df_books.drop(columns=["text", "tokens"], errors="ignore")

csv_saida = "gutenberg_books.csv"

# 5. Atualização acumulativa e correta no CSV base
if os.path.exists(csv_saida):
  df_base = pd.read_csv(csv_saida)
else:
  df_base = pd.DataFrame()

if not df_base.empty and not df_novos_dados.empty:
  df_base["book_id"] = df_base["book_id"].astype(str)
  df_novos_dados["book_id"] = df_novos_dados["book_id"].astype(str)

  # Concatena a base existente com os novos dados e remove duplicatas (mantém a versão mais recente)
  df_final = pd.concat([df_base, df_novos_dados]).drop_duplicates(
      subset=["book_id"], keep="last"
  )
else:
  df_final = df_novos_dados if not df_novos_dados.empty else df_base

# 6. Salva o resultado final preservando toda a base
df_final.to_csv(csv_saida, index=False, encoding="utf-8-sig")

Descompactando 147 arquivos .zip...
Erro ao descompactar gutenberg_harvest\2-h.zip: [Errno 22] Invalid argument: 'gutenberg_harvest\\245-h\\images\\157.jpg'
Total de arquivos HTML encontrados: 159
Livros válidos processados: 159
Extraindo e limpando os trechos...
Processo finalizado com sucesso! Registros salvos em 'gutenberg_books.csv' (Total de linhas: 312).


,book_id,title,author,trecho_1_clean,trecho_2_clean,tokens_clean
0,1342,Pride and Prejudice,"Austen, Jane, 1775-1817",drawing-room tea glad invite “ protested never...,colonel fitzwilliam called late evening might ...,"[pride, prejudice, project, gutenberg, preface..."
1,2701,"Moby Dick; Or, The Whale","Melville, Herman, 1819-1891",ship bound long perilous voyage — beyond storm...,distinctly perceive white mass quick intensity...,"[moby, dick, whale, project, gutenberg, moby-d..."
2,2554,Crime and Punishment,"Dostoyevsky, Fyodor, 1821-1881",head “ tell respectable luise ivanovna tell la...,finding nothing got drew deep breath reaching ...,"[crime, punishment, project, gutenberg, crime,..."


In [4]:
import nltk
import string

for pkg in ["punkt", "punkt_tab", "stopwords"]:
  try:
    nltk.download(pkg, quiet=True)
  except Exception as e:
    print(f"Aviso: não foi possível baixar {pkg}: {e}")

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

STOPWORDS_EN = set(stopwords.words("english"))
PUNCT = set(string.punctuation)


def tokenize_text(text):
  """Normaliza (minúsculas) e tokeniza o texto usando o NLTK."""
  return word_tokenize(text.lower(), language="english")


def remove_stopwords_punct(tokens):
  """Remove stopwords e pontuação da lista de tokens."""
  return [
      tok
      for tok in tokens
      if tok not in STOPWORDS_EN and not all(ch in PUNCT for ch in tok)
  ]


# Célula 7: Contagem de tokens brutos e limpos
df_books["n_tokens"] = df_books["tokens"].apply(len)
df_books["n_tokens_clean"] = df_books["tokens_clean"].apply(len)

# Visualiza as primeiras linhas com as contagens e metadados HTML extraídos
df_books[[
    "book_id",
    "title",
    "author",
    "n_tokens",
    "n_tokens_clean",
    "tokens_clean",
]].head()

,book_id,title,author,n_tokens,n_tokens_clean,tokens_clean
0,11,Alice's Adventures in Wonderland,"Carroll, Lewis, 1832-1898",35204,15443,"[alice, ’, adventures, wonderland, project, gu..."
1,1260,Jane Eyre: An Autobiography,"Brontë, Charlotte, 1816-1855",229840,97850,"[jane, eyre, project, gutenberg, jane, eyre, a..."
2,1342,Pride and Prejudice,"Austen, Jane, 1775-1817",152928,63239,"[pride, prejudice, project, gutenberg, preface..."
3,145,Middlemarch,"Eliot, George, 1819-1880",377168,165954,"[middlemarch, project, gutenberg, middlemarch,..."
4,2363,Desconhecido,Desconhecido,26078,10380,"[incognita, love, duty, reconcil, ’, novel, wi..."
